In [ ]:
# 0. 라이브러리 불러오기
import numpy as np
import pandas as pd
import tensorflow as tf

import tensorflow as tf
from tensorflow.keras import layers, models

import shap

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving X_rank3_tensor.npy to X_rank3_tensor.npy


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving y_labels.npy to y_labels.npy


In [ ]:
x_raw = np.load('X_rank3_tensor.npy')
y_raw = np.load('y_labels.npy')

x_raw.shape

(1765, 60, 10)

In [ ]:
print(y_raw[:10], y_raw.shape)

['ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP' 'ESTP'] (1765,)


In [ ]:

track_features = [
    'tempo', 'loudness', 'mode', 'danceability', 'energy',
    'speechiness', 'acousticness', 'liveness', 'happiness', 'instrumentalness'
]

numeric_features = [
    f for f in track_features
    if f not in ['mode', 'instrumentalness']
]
categorical_feature = ['mode']


x_train, x_test, y_train_str, y_test_str = train_test_split(
    x_raw, y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw

x_train_partial, x_val, y_train_partial_str, y_val_str = train_test_split(
    x_train, y_train_str,
    test_size=0.3,
    random_state=42,
    stratify=y_train_str
)

scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)


def data_preprocessed(x_data, is_train=False):
    (n_playlists, n_tracks, n_features) = x_data.shape

    x_2d = x_data.reshape(n_playlists * n_tracks, n_features)
    x_df = pd.DataFrame(x_2d, columns=track_features)

    real_song = x_df['tempo'] > 0

    if is_train:
        scaler.fit(x_df.loc[real_song, numeric_features])

    x_df.loc[real_song, numeric_features] = scaler.transform(
        x_df.loc[real_song, numeric_features]
    )

    if is_train:
        encoder.fit(x_df.loc[real_song, categorical_feature])

    mode_encoded = encoder.transform(x_df[categorical_feature])
    mode_encoded[~real_song] = 0 

    x_numeric = x_df[numeric_features].values
    x_2d_processed = np.concatenate([x_numeric, mode_encoded], axis=1)

    n_features_new = x_2d_processed.shape[1]
    x_processed = x_2d_processed.reshape(n_playlists, n_tracks, n_features_new)

    return x_processed

x_train_partial_prep = data_preprocessed(x_train_partial, is_train=True)
x_val_prep           = data_preprocessed(x_val,           is_train=False)
x_test_prep          = data_preprocessed(x_test,          is_train=False)

print("x_train shape (prep):", x_train_partial_prep.shape)

x_train shape (prep): (988, 60, 10)


In [ ]:
def mbti_to_vector(mbti: str):
    """
    [E(1)/I(0), N(1)/S(0), F(1)/T(0), P(1)/J(0)]
    """
    v = np.zeros(4, dtype=int)

    v[0] = 1 if mbti[0] == 'E' else 0
    v[1] = 1 if mbti[1] == 'N' else 0
    v[2] = 1 if mbti[2] == 'F' else 0
    v[3] = 1 if mbti[3] == 'P' else 0

    return v

def mbti_array_to_vectors(y_str_array):
    return np.stack([mbti_to_vector(m) for m in y_str_array], axis=0)

y_train_partial_bin = mbti_array_to_vectors(y_train_partial_str)
y_val_bin           = mbti_array_to_vectors(y_val_str)
y_test_bin          = mbti_array_to_vectors(y_test_str)

print("y_train shape (bin):", y_train_partial_bin.shape)

y_train shape (bin): (988, 4)


# cnn 모델

In [ ]:
import tensorflow.keras.backend as K
from tensorflow.keras import layers, models

n_tracks       = x_train_partial_prep.shape[1]
n_features_new = x_train_partial_prep.shape[2]

def build_model_deep_dense_weighted():

    inputs = layers.Input(shape=(n_tracks, n_features_new))

    # CNN block
    x = layers.Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)

    '''
    mask = layers.Lambda(make_mask, name="padding_mask")(inputs)
    pooled = layers.Lambda(masked_global_average_pooling, name="masked_global_avg")([x, mask])
    '''

    h = layers.Dense(128, activation='relu')(pooled)
    h = layers.Dropout(0.5)(h)
    h = layers.Dense(64, activation='relu')(h)
    h = layers.Dropout(0.3)(h)
    outputs = layers.Dense(4, activation='sigmoid')(h)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer='adam',
        loss=weighted_binary_crossentropy,  
        metrics=['accuracy']
    )
    return model

In [ ]:
# model = build_model_kernel5()
# model = build_model_filters_big()
model = build_model_deep_dense_weighted()
# model = build_model_more_dropout()

In [ ]:
EPOCHS = 60          
BATCH_SIZE = 32

interval = max(1, EPOCHS // 5)

#LLM으로 작성한 함수 
class IntervalLogger(tf.keras.callbacks.Callback):
    def __init__(self, interval, total_epochs):
        super().__init__()
        self.interval = interval
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        epoch_num = epoch + 1
        if (epoch_num % self.interval == 0) or (epoch_num == self.total_epochs):
            train_loss = logs.get('loss')
            train_acc = logs.get('accuracy')
            val_loss   = logs.get('val_loss')
            val_acc    = logs.get('val_accuracy')

            print(f"\n[Epoch {epoch_num}/{self.total_epochs}]")
            print(f" - train loss: {train_loss:.4f}  |  train acc: {train_acc:.4f}")
            if val_loss is not None and val_acc is not None:
                print(f" -  val  loss: {val_loss:.4f}  |   val acc: {val_acc:.4f}")

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,              
    restore_best_weights=True,
    verbose=1
)

# lr_scheduler 부분 - LLM으로 작성한 부분
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,        
    patience=3,        
    min_lr=1e-6,
    verbose=1
)

interval_logger = IntervalLogger(interval=interval, total_epochs=EPOCHS)

history = model.fit(
    x_train_partial_prep, y_train_partial_bin,
    validation_data=(x_val_prep, y_val_bin),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[interval_logger, early_stopping, lr_scheduler],  # lr_scheduler
    verbose=0  
)

y_pred_proba = model.predict(x_test_prep)

y_pred_bin = (y_pred_proba >= 0.5).astype(int)

dim_names = ['I/E', 'N/S', 'T/F', 'J/P']

for i in range(4):
    acc_i = (y_pred_bin[:, i] == y_test_bin[:, i]).mean()
    print(f"{dim_names[i]} accuracy: {acc_i:.4f}")

def vector_to_mbti(vec_bin):
    c0 = 'E' if vec_bin[0] == 1 else 'I'
    c1 = 'N' if vec_bin[1] == 1 else 'S'
    c2 = 'F' if vec_bin[2] == 1 else 'T'
    c3 = 'P' if vec_bin[3] == 1 else 'J'
    return c0 + c1 + c2 + c3

def vectors_to_mbti_array(y_bin_array):
    return np.array([vector_to_mbti(v) for v in y_bin_array])

y_pred_mbti = vectors_to_mbti_array(y_pred_bin)
y_true_mbti = np.array(y_test_str)

mbti_exact_acc = (y_pred_mbti == y_true_mbti).mean()
print(f"Final MBTI exact match accuracy: {mbti_exact_acc:.4f}")


Epoch 4: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.

Epoch 7: ReduceLROnPlateau reducing learning rate to 7.812500371073838e-06.
Epoch 8: early stopping
Restoring model weights from the end of the best epoch: 1.
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
I/E accuracy: 0.7960
N/S accuracy: 0.6941
T/F accuracy: 0.8102
J/P accuracy: 0.6317
Final MBTI exact match accuracy: 0.3286
